In [1]:
import logging
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"                   # sin mensajes informativos de MLflow
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"   # sin barras de progreso al guardar o cargar el modelo
# Permite importar el paquete `proyecto` sin instalarlo: el código vive en src/
sys.path.insert(0, str(Path.cwd().resolve().parent / "src"))

# 08 · Registro del modelo

En la optimización (notebook 06) hubo **muchas ejecuciones**, una por ensayo: eso es **tracking**, y responde *"¿qué
probamos?"*. Ahora hay **un modelo final**, validado una sola vez con la prueba (notebook 07), y lo que se guarda es otra
cosa: una **versión del modelo**, con su firma, un ejemplo de entrada, sus métricas de la prueba y un **alias**
(`champion`) que dice cuál es el vigente. El registro responde *"¿cuál es el modelo que usamos y en qué versión?"*.

| | Tracking (notebook 06) | Registro (este notebook) |
|---|---|---|
| Qué guarda | cada ensayo: parámetros y métricas | el modelo final, versionado |
| Cuántos hay | muchos (20 ensayos) | pocos (una versión por modelo que pasa la validación) |
| Para qué | comparar, reproducir y auditar la búsqueda | saber **cuál** es el modelo en uso y poder volver a una versión anterior |

> **Seguridad:** el modelo se guarda en formato *pickle*, que **puede ejecutar código al cargarse**. Solo se cargan modelos
> que registró el propio equipo, nunca uno de una fuente en la que no se confíe.

## 0. Conexión con el seguimiento

Desde el computador se lee el `.env`. Si no hay conexión con Databricks (no existe `.env`, no hay red o se agotó la cuota),
se usa MLflow **local** (`mlflow_local.db`, en la raíz del repositorio y fuera de Git): el registro local funciona igual. Es el
respaldo previsto por el curso. Cuando el equipo complete `.env`, este mismo notebook registra en Unity Catalog de
Databricks sin cambiar el código (el nombre pasa a ser `catálogo.esquema.modelo`).

In [2]:
import json
from datetime import datetime

import joblib
import mlflow
import numpy as np
import pandas as pd
from mlflow import MlflowClient
from mlflow.models import infer_signature

from proyecto.config import ID_COL, PROCESSED_DIR, REPORTS_DIR, ROOT, TARGET
from proyecto.features.engineering import ENGINEERED_FEATURES
from proyecto.models.tuning import _mlflow_name
from proyecto.tracking.mlflow_utils import configure_mlflow, registered_model_name, set_experiment
from proyecto.viz import fmt_num, table

pd.set_option("display.width", 170)
logging.getLogger("mlflow").setLevel(logging.ERROR)   # después de importar mlflow, que fija sus propios niveles
with warnings.catch_warnings(record=True) as caught:   # se muestra por qué se usó (o no) Databricks
    warnings.simplefilter("always")
    where = configure_mlflow()
for w in caught:
    print("Aviso:", w.message)
experiment = set_experiment("retiro-asociados-modelo-final")
MODEL_NAME = registered_model_name("retiro_asociados_rf")
print("MLflow:", where)
print("Experimento:", experiment)
print("Nombre del modelo en el registro:", MODEL_NAME)

MLflow: local
Experimento: retiro-asociados-modelo-final
Nombre del modelo en el registro: retiro_asociados_rf


## 1. El modelo final y lo que se sabe de él

El pipeline completo lo dejó el notebook 07 en `models/final_pipeline.joblib` (variables nuevas + preprocesamiento + Random
Forest, entrenado con los 155.258 asociados de entrenamiento). **Ese archivo no se versiona en Git** (pesa decenas de MB):
la fuente de verdad es el **registro**, y desde él se exportará el `model.joblib` de la app en el notebook 09.

Las cifras salen de los archivos del 07: `07_decisiones_congeladas.json` y `07_resultados.json`.

In [3]:
final_model = joblib.load(ROOT / "models" / "final_pipeline.joblib")
results = json.loads((REPORTS_DIR / "07_resultados.json").read_text(encoding="utf-8"))
decisions = json.loads((REPORTS_DIR / "07_decisiones_congeladas.json").read_text(encoding="utf-8"))
mark = (REPORTS_DIR / "prueba_abierta.txt").read_text(encoding="utf-8").strip()

print("Modelo:", results["modelo_final"], "· calibrado:", decisions["calibrar"])
print("Decisiones congeladas el:", decisions["fecha_y_hora"])
print("Prueba:", mark)
print(f"Tamaño del pipeline: {fmt_num((ROOT / 'models' / 'final_pipeline.joblib').stat().st_size / 1e6, 1)} MB")

Modelo: Random Forest · calibrado: False
Decisiones congeladas el: 2026-11-05T20:35:20
Prueba: Prueba abierta el 2026-11-05T20:36:06 · modelo Random Forest · 38.815 asociados. Una sola vez.
Tamaño del pipeline: 54,6 MB


### Las columnas de entrada

El modelo necesita **las columnas crudas que se conocen al decidir**. Las 5 variables nuevas (`in_risk_window`,
`is_dormant`, `savings_to_contributions`, `account_balance_trend`, `prog_savings_trend`) **no se piden**: el pipeline las
calcula adentro. Se comprueba que con solo esas columnas el modelo predice exactamente igual.

In [4]:
columns = json.loads((REPORTS_DIR / "04_seleccion_variables.json").read_text(encoding="utf-8"))["columns"]
INPUT_COLUMNS = [c for c in columns if c not in ENGINEERED_FEATURES]

train = joblib.load(PROCESSED_DIR / "train.joblib")
example = train[INPUT_COLUMNS].head(200).astype("float64")   # en la firma, todo numérico como decimal (admite vacíos)
same = np.allclose(final_model.predict_proba(train.drop(columns=[TARGET]).head(500))[:, 1],
                   final_model.predict_proba(train[INPUT_COLUMNS].head(500))[:, 1])
print(f"Columnas de entrada: {len(INPUT_COLUMNS)} (de las {len(columns)} que usa el modelo; las otras 5 se calculan adentro)")
print("Con solo esas columnas, las 500 predicciones de control son idénticas:", same)
assert same
print(INPUT_COLUMNS)

Columnas de entrada: 22 (de las 27 que usa el modelo; las otras 5 se calculan adentro)
Con solo esas columnas, las 500 predicciones de control son idénticas: True
['saldo_aportes', 'porc_meses_capitaliza_aportes', 'antig_meses', 'saldo_ahorro_prog', 'prom_saldo_ahorro_prog', 'rel_min-max_ahorro_prog', 'saldo_cta', 'prom_saldo_cta', 'rel_min-max_saldo_cta', 'prom_saldo_cartera', 'rel_min-max_saldo_cartera', 'porc_meses_publiturno', 'prom_trx', 'edad', 'femenino', 'ingresos', 'asalariado', 'ed_superior', 'zona_med', 'has_loan', 'has_prog_savings', 'has_account']


## 2. Qué se registra

Junto al modelo viajan:

- los **parámetros** y las **decisiones** (modelo, si se calibró, regla de decisión y supuestos de negocio);
- las **métricas de la prueba**, la misma tabla del notebook 07, con el prefijo `test_`;
- la **firma** (22 columnas de entrada y sus tipos; salida: probabilidad de cada clase) y un **ejemplo de entrada**;
- **etiquetas** para encontrarlo y auditarlo (cuándo se congelaron las decisiones, cuándo se abrió la prueba, unidad de
  partición);
- como **artefactos**, los archivos de decisiones y resultados del 07, para que cualquiera pueda auditar el modelo desde el
  registro;
- el **código del proyecto** (`src/proyecto`, con `code_paths`): el pipeline usa nuestras propias funciones (las variables
  nuevas), y sin ese código el modelo no se podría cargar en otro equipo ni en Databricks.

> **Nota técnica (del curso):** se usa `serialization_format="cloudpickle"`. El formato por defecto de MLflow rechaza un
> `Pipeline` con transformadores propios (error *untrusted types*).

In [5]:
params = {"modelo": results["modelo_final"], "calibrado": str(decisions["calibrar"]),
          **{f"param_{k}": v for k, v in decisions["parametros"].items()},
          "regla_de_decision": decisions["regla_de_decision"],
          "costo_contacto_millones": decisions["supuestos"]["costo_contacto"],
          "efectividad_retencion": decisions["supuestos"]["efectividad_retencion"],
          "columnas_de_entrada": len(INPUT_COLUMNS)}
test_metrics = {f"test_{_mlflow_name(k)}": float(v) for k, v in results["metricas_modelo"].items()}
test_metrics["test_beneficio_contactar_a_todos_por_1000"] = float(results["beneficio_contactar_a_todos_por_1000"])
for row in results["intervalos"]:
    key = _mlflow_name(f"ic_{row['qué']}_{row['métrica']}").replace("−", "menos")
    test_metrics[f"{key}_inf"] = float(row["IC 95 % inferior"])
    test_metrics[f"{key}_sup"] = float(row["IC 95 % superior"])
signature = infer_signature(example, final_model.predict_proba(example))

with mlflow.start_run(run_name="modelo-final-random-forest") as run:
    mlflow.log_params(params)
    mlflow.log_metrics(test_metrics)
    mlflow.set_tags({"etapa": "08_registro", "decisiones_congeladas": decisions["fecha_y_hora"],
                     "prueba_abierta": mark, "unidad_de_particion": ID_COL,
                     "supera_referencias_con_confianza": str(results["supera_referencias_con_confianza"])})
    mlflow.log_dict(decisions, "auditoria/07_decisiones_congeladas.json")
    mlflow.log_dict(results, "auditoria/07_resultados.json")
    info = mlflow.sklearn.log_model(
        final_model, name="model", signature=signature, input_example=example.head(5),
        serialization_format="cloudpickle", registered_model_name=MODEL_NAME,
        code_paths=[str(ROOT / "src" / "proyecto")],
    )
    run_id = run.info.run_id

client = MlflowClient()
version = max(int(v.version) for v in client.search_model_versions(f"name='{MODEL_NAME}'"))
print("Ejecución:", run_id)
print(f"Modelo registrado: {MODEL_NAME} · versión {version}")
print(f"Se registraron {len(params)} parámetros y {len(test_metrics)} métricas de la prueba")

Ejecución: 7bfca02731c243728892ec2e6c70223d
Modelo registrado: retiro_asociados_rf · versión 1
Se registraron 11 parámetros y 28 métricas de la prueba


Successfully registered model 'retiro_asociados_rf'.
Created version '1' of model 'retiro_asociados_rf'.


### ¿Lo registrado es lo que se midió?

Se leen de vuelta, **desde MLflow**, las métricas guardadas y se comparan con las del notebook 07. Si no coinciden, el
registro no estaría documentando lo que se evaluó.

In [6]:
logged = client.get_run(run_id).data.metrics
check = pd.DataFrame({
    "notebook 07": [results["metricas_modelo"][k] for k in ["PR-AUC", "AUC", "beneficio neto (por 1.000)", "Brier",
                                                            "saldo cubierto", "contactos (por 1.000)"]],
    "registrado en MLflow": [logged[f"test_{_mlflow_name(k)}"] for k in ["PR-AUC", "AUC", "beneficio neto (por 1.000)",
                                                                        "Brier", "saldo cubierto", "contactos (por 1.000)"]],
}, index=["PR-AUC", "AUC", "beneficio neto (por 1.000)", "Brier", "saldo cubierto", "contactos (por 1.000)"])
check["¿coinciden?"] = np.isclose(check["notebook 07"], check["registrado en MLflow"])
display(table(check, decimals=4))
assert check["¿coinciden?"].all()
signature_inputs = mlflow.models.get_model_info(info.model_uri).signature.inputs.input_names()
print(f"La firma registrada pide {len(signature_inputs)} columnas; coinciden con las de entrada: {signature_inputs == INPUT_COLUMNS}")

,notebook 07,registrado en MLflow,¿coinciden?
PR-AUC,"0,7004","0,7004",True
AUC,"0,8958","0,8958",True
beneficio neto (por 1.000),"33,6011","33,6011",True
Brier,"0,1014","0,1014",True
saldo cubierto,"0,9411","0,9411",True
contactos (por 1.000),"410,4083","410,4083",True


La firma registrada pide 22 columnas; coinciden con las de entrada: True


## 3. Poner el alias `champion`

Una **versión** es un número que no dice nada sobre su uso. El **alias** sí: `champion` es "el modelo vigente". Quien
consuma el modelo pide `models:/<nombre>@champion` y no necesita saber qué número de versión es. Si mañana hay una versión
mejor (por ejemplo, con la calibración isotónica que quedó como candidata en el 07), se registra, se valida y se **mueve el
alias**: nadie cambia su código. Si algo sale mal, se mueve de vuelta.

In [7]:
client.set_registered_model_alias(MODEL_NAME, "champion", str(version))
client.set_model_version_tag(MODEL_NAME, str(version), "decisiones_congeladas", decisions["fecha_y_hora"])
client.set_model_version_tag(MODEL_NAME, str(version), "validacion", "prueba abierta una sola vez (notebook 07)")
client.update_model_version(MODEL_NAME, str(version), description=(
    "Random Forest (200 árboles, profundidad 14, hojas >= 15) que estima la probabilidad de que un asociado de Confiar "
    "radique la solicitud de retiro total. Decide con la regla de valor esperado: contactar si p x 0,2 x saldo de aportes "
    "> 0,03 millones. Prueba: PR-AUC 0,700; beneficio 33,60 millones por 1.000 asociados (+14,81 sobre contactar a todos)."))
champion = client.get_model_version_by_alias(MODEL_NAME, "champion")
print(f"alias champion → versión {champion.version}  (ejecución {champion.run_id[:8]}…)")
aliases = client.get_registered_model(MODEL_NAME).aliases            # {alias: versión}
versions = pd.DataFrame([{"versión": int(v.version),
                          "alias": ", ".join(a for a, ver in aliases.items() if str(ver) == str(v.version)) or "—",
                          "ejecución": v.run_id[:8], "estado": v.status}
                         for v in client.search_model_versions(f"name='{MODEL_NAME}'")]).set_index("versión")
display(versions)

alias champion → versión 1  (ejecución 7bfca027…)


,alias,ejecución,estado
versión,,,
1,champion,7bfca027,READY


## 4. Comprobación de ida y vuelta

Un registro que no se puede **recuperar** no sirve. Se carga el modelo **desde el registro, por su alias**, y se comprueba
que predice **exactamente igual** que el objeto que se evaluó en el notebook 07. Se usan 1.000 asociados del entrenamiento
(la prueba no se vuelve a usar).

In [8]:
from_registry = mlflow.sklearn.load_model(f"models:/{MODEL_NAME}@champion")
sample = train[INPUT_COLUMNS].sample(1000, random_state=0)
expected = final_model.predict_proba(sample)
obtained = from_registry.predict_proba(sample)
round_trip = bool(np.allclose(expected, obtained))
print(f"Las 1.000 predicciones coinciden: {round_trip} (diferencia máxima {np.abs(expected - obtained).max():.2e})")
assert round_trip, "El modelo del registro no coincide con el que se evaluó"

Las 1.000 predicciones coinciden: True (diferencia máxima 3.33e-16)


**Lectura.** El modelo que quedó en el registro es, bit a bit en sus predicciones, el mismo que se evaluó con la prueba: lo
que dice la ficha del 07 vale para lo que está registrado.

## 5. Guardar la constancia

Queda en `reports/08_registro.json` (y se resume en `reports/08_registro.md`): dónde se registró, qué versión, qué alias, con
qué decisiones y si la ida y vuelta funcionó. La app **no consulta el registro**: trabaja con el `model.joblib` que se exporta
desde `champion` en el notebook 09.

In [9]:
record = {
    "fecha_y_hora": datetime.now().isoformat(timespec="seconds"),
    "donde": where, "experimento": experiment,
    "modelo_registrado": MODEL_NAME, "version": version, "alias": "champion", "ejecucion": run_id,
    "uri_del_modelo": f"models:/{MODEL_NAME}@champion",
    "columnas_de_entrada": INPUT_COLUMNS,
    "ida_y_vuelta_ok": round_trip,
    "modelo": results["modelo_final"], "parametros": decisions["parametros"], "calibrado": decisions["calibrar"],
    "regla_de_decision": decisions["regla_de_decision"],
    "decisiones_congeladas": decisions["fecha_y_hora"], "prueba": mark,
    "metricas_prueba": {k: results["metricas_modelo"][k] for k in ["PR-AUC", "AUC", "beneficio neto (por 1.000)", "Brier"]},
}
(REPORTS_DIR / "08_registro.json").write_text(json.dumps(record, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print(json.dumps({k: record[k] for k in ["donde", "experimento", "modelo_registrado", "version", "alias", "ida_y_vuelta_ok",
                                         "decisiones_congeladas"]}, indent=2, ensure_ascii=False))

{
  "donde": "local",
  "experimento": "retiro-asociados-modelo-final",
  "modelo_registrado": "retiro_asociados_rf",
  "version": 1,
  "alias": "champion",
  "ida_y_vuelta_ok": true,
  "decisiones_congeladas": "2026-11-05T20:35:20"
}


## 6. Conclusiones

1. **El modelo quedó registrado con versión y alias.** `retiro_asociados_rf`, alias `champion`, en MLflow local (sin `.env`
   de Databricks). Con el `.env` completo, el mismo notebook lo registra en Unity Catalog.
2. **Viaja con todo lo necesario para confiar en él:** parámetros, regla de decisión y supuestos, las métricas de la prueba
   (comprobadas contra el notebook 07), la firma con las 22 columnas de entrada, un ejemplo de entrada, las decisiones
   congeladas y los resultados como artefactos de auditoría, y el código del proyecto para cargarlo en cualquier equipo.
3. **Lo registrado es lo evaluado:** la ida y vuelta por alias reproduce exactamente las predicciones del modelo del 07.
4. **Tracking y registro cumplen papeles distintos:** el experimento `retiro-asociados-optimizacion` guarda los 20 ensayos de
   la búsqueda (qué se probó); el registro guarda la versión que se usa (qué se despliega). Si la calibración isotónica u
   otra mejora se valida con datos nuevos, se registra como versión 2 y se mueve el alias.

**Siguiente:** `09_inferencia.ipynb`: exportar `models/model.joblib` desde `champion`, escribir el esquema de entrada
(`models/input_schema.json`) y comprobar que la función de predicción rechaza los datos imposibles. Con eso la app ya puede
cargar el modelo.